# 复现 notebook（replicate_all）

**目的**：一键复现第一阶段目前的所有结果。本 notebook 的代码**直接抽取自你原始的 notebook 单元**（product_character / outsourcing_analysis / descriptive_analysis / coverage / sample），未做重写，只做了三处最小改动：
1. 修了 `outsourcing_analysis` 里 `to_stata(..., index=False)` 的参数 bug（应为 `write_index=False`）；
2. 加了 glue：把 `firm_aggregate_table` 另存为 `summary.dta`（`descriptive_analysis`/`sample` 读取的就是它）；从 lenth9 切出 `lenth9_18.dta`；
3. **改了路径**（见下），让代码与数据分离。

## 路径约定（代码 git 共享，数据只在 VM）

| | 位置 | 说明 |
|---|---|---|
| **代码** `CODE` | `G:\Kuangyu_Temp\Outsource\Empirical1\` | git 同步，本地/VM 两边共享 |
| **生成数据** `DATA` | `G:\Kuangyu_Temp\Outsource\replicate\` | 本流程产出的**全部**数据，只在 VM，不进 git |
| **已有输入** `SRC` | `G:\Kuangyu_Temp\Outsource\` | `full_product_similarity.dta`、`io_table_lite.dta`（保持原位） |
| **原始数据** `RAW` | `G:\Kuangyu_Temp\single_product\1718_total_cleaned_by_year1.dta` | 最原始交易数据 |

每段开头都会 `os.chdir(DATA)`，所以所有相对读写都落在 `DATA`；只有 `SRC`/`RAW` 用绝对路径。

**起点**：从**最原始文件** `1718_total_cleaned_by_year1.dta` 开始。其中**超大原始文件的 collapse 交给 Stata**（`database.do`，由本 notebook 调用 VM 的 StataMP-64 运行），避免 pandas/VS Code 打不开大文件；9 位码标准化(`to_nine`)及之后在 Python 里跑。

## 运行顺序与产出
| 段 | 来源 | 读入 | 产出 |
|---|---|---|---|
| §0a | database.do (Stata) | 1718_total_cleaned_by_year1.dta | lenth15.dta |
| §0b | to_nine | lenth15.dta | lenth9.dta |
| §1 | product_character | lenth9.dta | firm_product_year_level.dta, product_characteristics.dta |
| §2 | product_character | firm_product_year_level, similarity | **full_data.dta**, firm_year_summary.dta |
| §3 | outsourcing_analysis | lenth9.dta | firm_aggregate_table.dta → **summary.dta** |
| §4 | descriptive_analysis | summary.dta | 描述统计 + 图 |
| §5 | coverage | full_data, similarity | coverage_summary.csv + 相似度分布图 |
| §6 | sample | summary.dta, lenth9_18.dta, io_table_lite.dta | similarity.dta（企业级余弦）|

> ⚠️ **口径提醒（照你现有代码，未改）**：§2 的 `full_data` 主产品用 **total_output** 最大；§3 的外包强度用**投入侧**口径（旧版，中介≈12.46%），与 §2 里 min 口径不同——这是你当前代码本来就并存的两套口径，本 notebook 忠实保留。若要统一口径请告诉我。
> ⚠️ **内存**：§1 与 §3 都会读 ~4.65 亿行的 lenth9，建议在 VM 大内存上跑。
> ⚠️ **前置输入**：`SRC` 下需有 `full_product_similarity.dta` 与 `io_table_lite.dta`；`lenth9_18.dta` 现由 §0b 自动生成，无需预先准备。

## §0a　database.do（Stata）：原始交易 → lenth15

超大原始文件的 collapse 在 **Stata** 里做（VM 能开、精确、无分块问题），notebook 通过 `StataMP-64 /e do` 调用。
产出 `lenth15.dta`。**改 `DO_DATABASE` 为 VM 上 database.do 的实际路径。**

In [ ]:
# [replicate paths] 代码在 Empirical1(git 共享)；生成数据全部落到 DATA
from pathlib import Path
import os
DATA = Path(r'G:\Kuangyu_Temp\Outsource\replicate')   # 所有生成数据（只在 VM）
SRC  = Path(r'G:\Kuangyu_Temp\Outsource')              # 已有输入数据（similarity / io_table）
DATA.mkdir(exist_ok=True)
os.chdir(DATA)

CODE = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1')     # 代码（git 同步，两边共享）
RAW  = Path(r'G:\Kuangyu_Temp\single_product\1718_total_cleaned_by_year1.dta')

import subprocess
stata_exe   = r"C:/Program Files/Stata17/StataMP-64.exe"
DO_DATABASE = str(CODE / 'replicate' / 'database.do')

print('CODE (代码, git):', CODE)
print('DATA (生成数据) :', DATA)
print('SRC  (已有输入) :', SRC)
print('\nrunning database.do in Stata ...')
proc = subprocess.run([stata_exe, "/e", "do", DO_DATABASE],
                      capture_output=True, text=True, errors="ignore")
print('Stata return code:', proc.returncode, '(0 = 正常)')
print('产出应为', DATA / 'lenth15.dta')

## §0b　to_nine：15 位码 → 9 位码（→ lenth9）

代码来自你原 `to_nine.ipynb`（仅改路径）。第一格为环境与导入；第二格做 9 位码层级判断、两侧企业取交集，读 `lenth15.dta` → 产出 `lenth9.dta`。末尾一格 glue 从 lenth9 切出 2018 子集 `lenth9_18.dta`，供 §6 使用。

In [ ]:
%reset -f
# [replicate paths] 代码在 Empirical1(git 共享)；生成数据全部落到 DATA
from pathlib import Path
import os
DATA = Path(r'G:\Kuangyu_Temp\Outsource\replicate')   # 所有生成数据（只在 VM）
SRC  = Path(r'G:\Kuangyu_Temp\Outsource')              # 已有输入数据（similarity / io_table）
DATA.mkdir(exist_ok=True)
os.chdir(DATA)

from IPython import get_ipython
import pandas as pd
import gc
import os
import numpy as np
import seaborn as sns
from collections import Counter
import subprocess
gc.collect()
import warnings
warnings.filterwarnings('ignore')
stata_exe = "C:/Program Files/Stata17/StataMP-64.exe"

In [ ]:
os.chdir(DATA)
df1 = pd.read_stata('lenth15.dta')
df1['product_id'] = df1['product_id'].astype(str)
df1['firm_id'] = df1['firm_id'].astype(str)
df2 = df1[df1['is_output']==1]#产出品
df3 = df1[df1['is_output']==0]#投入品

df_output = df2
df_input = df3

print('产出表')
print('Firm Count', df_output['firm_id'].nunique())
print('Product_count', df_output['product_id'].nunique())
print('Value', df_output['v'].sum()/100000000)
      
print('投入表')
print('Firm Count', df_input['firm_id'].nunique())
print('Product_count', df_input['product_id'].nunique())
print('Value', df_input['v'].sum()/100000000)

print(df3['product_id'].nunique())
print(df2['product_id'].nunique())
print(df1['product_id'].nunique())

# Step1：去掉1，3位的
df2['product_id_9'] = df2['product_id'].str[:9]
df3['product_id_9'] = df3['product_id'].str[:9]

def is_high_level(code):
    return(
        code[1:] == '0'*(len(code)-1) or
        code[3:] == '0'*(len(code)-3)
    )
    
df_high = df2[df2['product_id'].apply(is_high_level)]
df_low = df2[~df2['product_id'].apply(is_high_level)]

print('product_id')
print(df2['product_id'].nunique())
print("high level",df_high['product_id'].nunique())
print("low level",df_low['product_id'].nunique())

print('v')
print(df2['v'].sum()/100000000)
print(df_high['v'].sum()/100000000)
print(df_low['v'].sum()/100000000)

print('firm_id')
print(df2['firm_id'].nunique())
print(df_high['firm_id'].nunique())
print(df_low['firm_id'].nunique())


# Step2： 筛选5、7位
df_low_group = df_low.groupby('product_id')['v'].sum().reset_index()

df_low_group['product_id'] = df_low_group['product_id'].astype(str)
df_low_group['product_id_5'] = df_low_group['product_id'].str[:5]
df_low_group['product_id_7'] = df_low_group['product_id'].str[:7]
df_low_group['product_id_9'] = df_low_group['product_id'].str[:9]


#截取前七位码，如果没有重复的，那么说明该产品至多到7位
counts_1 = df_low_group.groupby('product_id_7')['product_id_9'].nunique()
single_prefixes_1 = counts_1[counts_1 == 1].index
#同理，梳理五位码
counts_2 = df_low_group.groupby('product_id_5')['product_id_7'].nunique()
single_prefixes_2 = counts_2[counts_2 == 1].index

filtered_df_1 = df_low_group[df_low_group['product_id_7'].isin(single_prefixes_1)]
filtered_df_2 = df_low_group[df_low_group['product_id_5'].isin(single_prefixes_2)]


#查询各位码中有多少个重复的
five_in_seven = [item for item in single_prefixes_1 if item.endswith('00')]#这里包含了全部96个的5位码
seven_in_seven = [item for item in single_prefixes_1 if not item.endswith('00')]
#把筛选出7位码的
print(len(seven_in_seven))
for i in five_in_seven:
    if i[:-2] in single_prefixes_2:
        seven_in_seven.append(i)
print(len(five_in_seven))
print(len(seven_in_seven))

seven_in_seven = [i + '00' for i in seven_in_seven]

#把其他的9位码整出来
product_id = df_low_group['product_id_9'].drop_duplicates().tolist()
product_id_9 = [item for item in product_id if not item.endswith('00')]

print(len(product_id_9))

product_id_9_final = product_id_9+seven_in_seven
print(len(product_id_9_final))

# Step3： 完成9位码的筛选
df_output = df2[df2['product_id_9'].isin(product_id_9_final)]
print(df_output['product_id'].nunique())
df_output = df_output.drop(columns={'product_id'})
df_output = df_output.rename(columns={'product_id_9':'product_id'})
df_output = df_output.groupby(['firm_id', 'product_id', 'is_output','year'])['v'].sum().reset_index()

df_input = df3[df3['product_id_9'].isin(product_id_9_final)]
print(df_input['product_id'].nunique())
df_input= df_input.drop(columns={'product_id'})
df_input = df_input.rename(columns={'product_id_9':'product_id'})
df_input = df_input.groupby(['firm_id', 'product_id', 'is_output','year'])['v'].sum().reset_index()

output_firm_list = df_output['firm_id'].drop_duplicates().tolist()
input_firm_list = df_input['firm_id'].drop_duplicates().tolist()
firm_list = set(output_firm_list)&set(input_firm_list)

print('firm')
print(df_input['firm_id'].nunique())
print(df_output['firm_id'].nunique())
df_input['firm_id'] = df_input['firm_id'].astype(str)
df_output['firm_id'] = df_output['firm_id'].astype(str)

print("output_value",df_output['v'].sum()/100000000)
print("input_value", df_input['v'].sum()/100000000)
print("all_value", df_output['v'].sum()/100000000+df_input['v'].sum()/100000000)

df_output = df_output[df_output['firm_id'].isin(firm_list)]
df_input = df_input[df_input['firm_id'].isin(firm_list)]
df1 = pd.concat([df_input, df_output])

print('产出表')
print('Firm Count', df_output['firm_id'].nunique())
print('Product_count', df_output['product_id'].nunique())
print('Value', df_output['v'].sum()/100000000)
      
print('投入表')
print('Firm Count', df_input['firm_id'].nunique())
print('Product_count', df_input['product_id'].nunique())
print('Value', df_input['v'].sum()/100000000)
df1.to_stata('lenth9.dta', write_index=False)

In [ ]:
# [replicate glue] 生成 lenth9_18.dta（lenth9 的 2018 子集），供 §6 sample 使用。
# 原代码读的是预先存在的 lenth9_18.dta；这里直接从本流程刚建好的 lenth9 切出来，保证上下游一致。
df1[df1['year'] == 2018].to_stata(DATA / 'lenth9_18.dta', write_index=False)
print('saved', DATA / 'lenth9_18.dta')

## §1　product_character：firm_product_year_level + product_characteristics

代码来自你原 `product_character.ipynb`（未改）。读 `lenth9`；产出/投入按 firm×product×year 求和，`outsourcing_value = min(投入,产出)`、`production_value = 产出 − 外包`、`outsourcing_percen`；存 `firm_product_year_level.dta`，再汇总产品特征 `product_characteristics.dta`。

In [ ]:
"""
产品级别数据整理脚本
从原始交易数据生成：
1. 企业-产品-年份级别的产出和外包数据
2. 产品特征数据
"""

import pandas as pd
import numpy as np

print("=" * 80)
print("产品级别数据整理")
print("=" * 80)

# ============================================================================
# 步骤1：读取原始交易数据
# ============================================================================
print("\n[1] 读取原始交易数据...")

# 请替换为你的实际数据文件路径
DATA_PATH = str(DATA/'lenth9.dta')  # 或 .csv

try:
    if DATA_PATH.endswith('.dta'):
        df = pd.read_stata(DATA_PATH)
    elif DATA_PATH.endswith('.csv'):
        df = pd.read_csv(DATA_PATH)
    else:
        raise ValueError("不支持的文件格式")
   
    print(f"✓ 数据读取成功")
    print(f"  原始记录数: {len(df):,}")
    print(f"  列名: {list(df.columns)}")
except Exception as e:
    print(f"✗ 数据读取失败: {e}")
    exit(1)

# ============================================================================
# 步骤2：数据清洗和准备
# ============================================================================
print("\n[2] 数据清洗...")

# 确保只使用year列（如果存在date列，提取year）
if 'date' in df.columns and 'year' not in df.columns:
    print("  从date列提取year...")
    df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
    # 如果date是字符串格式如"2017-01"，提取前4位
    if df['year'].isna().all():
        df['year'] = df['date'].astype(str).str[:4].astype(int, errors='ignore')

# 检查必需列
required_cols = ['year', 'firm_id', 'product_id', 'is_output', 'v']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    print(f"✗ 缺少必需列: {missing_cols}")
    exit(1)

# 删除缺失值
original_len = len(df)
df = df.dropna(subset=required_cols)
print(f"  删除缺失值: {original_len - len(df):,} 行")

# 删除负值交易
negative_count = (df['v'] < 0).sum()
if negative_count > 0:
    print(f"  删除负值交易: {negative_count:,} 行")
    df = df[df['v'] >= 0]

print(f"✓ 清洗后记录数: {len(df):,}")

# ============================================================================
# 步骤3：计算企业-产品-年份级别的产出和投入
# ============================================================================
print("\n[3] 计算企业-产品-年份级别的汇总数据...")

# 分别计算产出和投入
output_df = df[df['is_output'] == 1].groupby(['year', 'firm_id', 'product_id'])['v'].sum().reset_index()
output_df.columns = ['year', 'firm_id', 'product_id', 'total_output']

input_df = df[df['is_output'] == 0].groupby(['year', 'firm_id', 'product_id'])['v'].sum().reset_index()
input_df.columns = ['year', 'firm_id', 'product_id', 'total_input']

print(f"  产出记录: {len(output_df):,}")
print(f"  投入记录: {len(input_df):,}")

# ============================================================================
# 步骤4：合并产出和投入，计算外包值和自产值
# ============================================================================
print("\n[4] 计算外包值和自产值...")

# 合并产出和投入（以产出为主）
product_level = output_df.merge(
    input_df,
    on=['year', 'firm_id', 'product_id'],
    how='left'
)

# 填充没有投入的产品（全部自产）
product_level['total_input'] = product_level['total_input'].fillna(0)

# 计算外包值：min(投入, 产出)
product_level['outsourcing_value'] = np.minimum(
    product_level['total_input'],
    product_level['total_output']
)

# 计算自产值：产出 - 外包
product_level['production_value'] = (
    product_level['total_output'] - product_level['outsourcing_value']
)

# 计算外包比例
product_level['outsourcing_percen'] = (
    product_level['outsourcing_value'] / product_level['total_output']
)

# 处理除以0的情况
product_level['outsourcing_percen'] = product_level['outsourcing_percen'].fillna(0)

print(f"✓ 企业-产品-年份级别数据: {len(product_level):,} 条")
print(f"\n描述性统计:")
print(product_level[['total_output', 'total_input', 'outsourcing_value',
                     'production_value', 'outsourcing_percen']].describe())

# ============================================================================
# 步骤5：保存第一份表（企业-产品-年份级别）
# ============================================================================
print("\n[5] 保存企业-产品-年份级别数据...")

# 选择并重命名列
firm_product_table = product_level[[
    'year', 'firm_id', 'product_id',
    'total_output', 'outsourcing_value', 'production_value',
    'outsourcing_percen'
]].copy()

# 保存为Stata格式
OUTPUT_FILE_1 = 'firm_product_year_level.dta'
firm_product_table.to_stata(OUTPUT_FILE_1, write_index=False)
print(f"✓ 已保存: {OUTPUT_FILE_1}")
print(f"  包含 {len(firm_product_table):,} 条记录")


# ============================================================================
# 步骤6：计算产品特征数据（不分年份，汇总所有年份）
# ============================================================================
print("\n[6] 计算产品特征数据（汇总所有年份）...")

# 按product_id汇总，不分年份
product_characteristics = product_level.groupby('product_id').agg({
    'firm_id': 'nunique',  # 生产该产品的企业数（跨年份去重）
    'total_output': 'sum',  # 总产出（所有年份）
    'outsourcing_value': 'sum',  # 总外包（所有年份）
    'production_value': 'sum',  # 总自产（所有年份）
    'year': lambda x: list(x.unique())  # 该产品出现的年份
}).reset_index()

# 重命名列
product_characteristics.columns = [
    'product_id',
    'num_firms',  # 生产该产品的企业数（popularity）
    'total_output',
    'total_outsourcing',
    'total_production',
    'years_present'  # 该产品出现的年份列表
]

# 计算产品出现的年份数
product_characteristics['num_years'] = product_characteristics['years_present'].apply(len)

# 删除years_present列（如果不需要的话）
product_characteristics = product_characteristics.drop('years_present', axis=1)

# 计算总外包强度
product_characteristics['outsourcing_intensity'] = (
    product_characteristics['total_outsourcing'] /
    product_characteristics['total_output']
)

# 处理除以0
product_characteristics['outsourcing_intensity'] = (
    product_characteristics['outsourcing_intensity'].fillna(0)
)

# 添加其他有用的特征
# 平均每家企业的产出
product_characteristics['avg_output_per_firm'] = (
    product_characteristics['total_output'] /
    product_characteristics['num_firms']
)

# 平均每年的产出
product_characteristics['avg_output_per_year'] = (
    product_characteristics['total_output'] /
    product_characteristics['num_years']
)

# 计算产品的总产出排名（越小越重要）
product_characteristics['output_rank'] = (
    product_characteristics['total_output']
    .rank(ascending=False, method='dense')
    .astype(int)
)

print(f"✓ 产品特征数据: {len(product_characteristics):,} 个产品")
print(f"\n产品受欢迎程度分布:")
print(product_characteristics['num_firms'].describe())

print(f"\n外包强度分布:")
print(product_characteristics['outsourcing_intensity'].describe())

print(f"\n产品出现年份数分布:")
print(product_characteristics['num_years'].value_counts().sort_index())

# ============================================================================
# 步骤7：保存第二份表（产品特征）
# ============================================================================
print("\n[7] 保存产品特征数据...")

# 按产品受欢迎程度排序
product_characteristics = product_characteristics.sort_values(
    'num_firms',
    ascending=False
)

# 保存为Stata格式
OUTPUT_FILE_2 = 'product_characteristics.dta'
product_characteristics.to_stata(OUTPUT_FILE_2, write_index=False)
print(f"✓ 已保存: {OUTPUT_FILE_2}")
print(f"  包含 {len(product_characteristics):,} 个产品")



# ============================================================================
# 步骤8：生成汇总报告
# ============================================================================
print("\n" + "=" * 80)
print("数据整理完成！")
print("=" * 80)

print("\n【输出文件】")
print(f"1. firm_product_year_level.dta / .csv")
print(f"   - 企业-产品-年份级别数据")
print(f"   - {len(firm_product_table):,} 条记录")
print(f"   - 列: year, firm_id, product_id, total_output,")
print(f"         outsourcing_value, production_value, outsourcing_percen")

print(f"\n2. product_characteristics.dta / .csv")
print(f"   - 产品特征数据（不分年份）")
print(f"   - {len(product_characteristics):,} 个产品")
print(f"   - 列: product_id, num_firms, total_output,")
print(f"         total_outsourcing, total_production, outsourcing_intensity,")
print(f"         avg_output_per_firm, avg_output_per_year, output_rank, num_years")

print("\n【关键统计】")
print(f"年份范围: {product_level['year'].min()} - {product_level['year'].max()}")
print(f"企业数: {product_level['firm_id'].nunique():,}")
print(f"产品数: {product_level['product_id'].nunique():,}")
print(f"企业-产品-年份记录数: {len(product_level):,}")

print("\n【前10个最受欢迎的产品】")
top_products = product_characteristics.nlargest(10, 'num_firms')[
    ['product_id', 'num_firms', 'total_output', 'outsourcing_intensity', 'num_years']
]
print(top_products.to_string(index=False))

print("\n【外包强度最高的10个产品（至少10家企业生产）】")
high_outsourcing = product_characteristics[
    product_characteristics['num_firms'] >= 10
].nlargest(10, 'outsourcing_intensity')[
    ['product_id', 'num_firms', 'outsourcing_intensity', 'num_years']
]
print(high_outsourcing.to_string(index=False))

print("\n" + "=" * 80)
print("所有数据已成功整理！")
print("=" * 80)

## §2　product_character：full_data + firm_year_summary

代码来自你原 `product_character.ipynb`（未改）。企业层汇总（外包强度、`is_intermediary`>0.90、`is_outsourcing`≥0.01）→ `firm_year_summary.dta`；主产品 = **total_output 最大**；合并 similarity（主产品自身=1）→ `full_data.dta`。

In [ ]:
import pandas as pd
import numpy as np
import os
os.chdir(DATA)



df = pd.read_stata("firm_product_year_level.dta")
df["year"] = df["year"].astype(int)

# ============================================================
# 公司-年份层面汇总
# ============================================================
firm_summary = df.groupby(["year", "firm_id"]).agg(
    firm_total_output    = ("total_output",      "sum"),
    firm_total_outsource = ("outsourcing_value", "sum"),
    n_products           = ("product_id",        "count"),
).reset_index()

firm_summary["outsourcing_intensity"] = (
    firm_summary["firm_total_outsource"] / firm_summary["firm_total_output"]
).fillna(0)

firm_summary["is_intermediary"] = (firm_summary["outsourcing_intensity"] > 0.9).astype(int)
firm_summary["is_outsourcing"]  = (firm_summary["outsourcing_intensity"] >= 0.01).astype(int)

firm_summary.to_stata("firm_year_summary.dta", write_index=False)

# ============================================================
# 确定主营产品（firm-year 内 total_output 最大，并列取 product_id 最小）
# ============================================================
df_sorted = df.sort_values(["year", "firm_id", "total_output", "product_id"],
                           ascending=[True, True, False, True])
main_product = (
    df_sorted
    .groupby(["year", "firm_id"], as_index=False)
    .first()[["year", "firm_id", "product_id", "total_output"]]
    .rename(columns={"product_id": "main_product", "total_output": "main_product_output"})
)

df = df.merge(main_product, on=["year", "firm_id"], how="left")
df["is_main"] = (df["product_id"] == df["main_product"]).astype(int)

# ============================================================
# 合并公司特征
# ============================================================
df = df.merge(firm_summary, on=["year", "firm_id"], how="left")

df["sales_percen"]        = df["total_output"] / df["firm_total_output"]
df["sales_relative_main"] = df["total_output"] / df["main_product_output"]

# ============================================================
# 合并相似度（对称矩阵，合并两个方向后去重）
# ============================================================
sim = pd.read_stata(str(SRC/'full_product_similarity.dta'))

sim_dir1 = sim.rename(columns={"product_1": "product_id", "product_2": "main_product"})
sim_dir2 = sim.rename(columns={"product_2": "product_id", "product_1": "main_product"})
sim_lookup = (
    pd.concat([sim_dir1, sim_dir2])
    .drop_duplicates(subset=["product_id", "main_product"])
    .reset_index(drop=True)
)

df = df.merge(sim_lookup, on=["product_id", "main_product"], how="left")

# 主营产品与自身相似度定义为 1
df.loc[df["is_main"] == 1, "input_similarity"]  = 1.0
df.loc[df["is_main"] == 1, "output_similarity"] = 1.0

# ============================================================
# 整理列顺序并保存
# ============================================================
cols = [
    "year", "firm_id", "product_id",
    "total_output", "outsourcing_value", "production_value",
    "outsourcing_percen", "sales_percen", "sales_relative_main",
    "is_main", "main_product", "main_product_output",
    "input_similarity", "output_similarity",
    "firm_total_output", "firm_total_outsource", "n_products",
    "outsourcing_intensity", "is_intermediary", "is_outsourcing",
]
df = df[cols].sort_values(["year", "firm_id", "total_output"],
                          ascending=[True, True, False]).reset_index(drop=True)

df.to_stata("full_data.dta", write_index=False)

## §3　outsourcing_analysis：firm_aggregate_table → summary.dta

代码来自你原 `outsourcing_analysis.ipynb`（仅修一处 `to_stata` 参数）+ 一格 glue。读 `lenth9` 建企业汇总表；**外包强度用投入侧口径（旧，中介≈12.46%）**；另存为 `summary.dta` 供 §4/§6 读取。

In [ ]:
%reset -f
# [replicate paths] 代码在 Empirical1(git 共享)；生成数据全部落到 DATA
from pathlib import Path
import os
DATA = Path(r'G:\Kuangyu_Temp\Outsource\replicate')   # 所有生成数据（只在 VM）
SRC  = Path(r'G:\Kuangyu_Temp\Outsource')              # 已有输入数据（similarity / io_table）
DATA.mkdir(exist_ok=True)
os.chdir(DATA)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')
os.chdir(DATA)
# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置绘图风格
sns.set_style("whitegrid")
sns.set_palette("husl")

print("=" * 80)
print("企业外包行为描述性分析")
print("=" * 80)

In [ ]:


# ============================================================================
# Step 1: 数据读取和清洗
# ============================================================================
print("\n[Step 1] 读取和清洗数据...")

# 读取数据
df = pd.read_stata(str(DATA/'lenth9.dta'))
print(f"原始数据规模: {len(df):,} 行")



# 基本统计
print(f"\n企业数量: {df['firm_id'].nunique():,}")
print(f"产品数量: {df['product_id'].nunique():,}")
print(f"总交易额: {df['v'].sum()/1e9:.2f} 十亿元")

# ============================================================================
# Step 2: 识别外包产品
# ============================================================================
print("\n[Step 2] 识别外包产品...")

# 对每个企业-产品-日期组合，判断是否同时有买入和卖出
firm_product_summary = df.groupby(['firm_id', 'product_id', 'year']).agg({
    'is_output': lambda x: (x == 1).any() and (x == 0).any(),  # 既买又卖
    'v': 'sum'
}).reset_index()

firm_product_summary.columns = ['firm_id', 'product_id', 'year', 'is_outsourcing', 'total_value']

# 合并回原数据
df = df.merge(
    firm_product_summary[['firm_id', 'product_id', 'year', 'is_outsourcing']], 
    on=['firm_id', 'product_id', 'year'],
    how='left'
)

print(f"外包产品-企业-年份组合数: {df[df['is_outsourcing']].groupby(['firm_id', 'product_id', 'year']).ngroups:,}")

# ============================================================================
# Step 3: 构建企业层面汇总表
# ============================================================================
print("\n[Step 3] 构建企业层面汇总表...")

# 分别计算产出和投入
output_df = df[df['is_output'] == 1].copy()
input_df = df[df['is_output'] == 0].copy()

# 3.1 总产出和总投入
firm_totals = df.groupby(['firm_id', 'year']).agg({
    'v': lambda x: x[df.loc[x.index, 'is_output'] == 1].sum() if len(x[df.loc[x.index, 'is_output'] == 1]) > 0 else 0,
}).reset_index()
firm_totals.columns = ['firm_id', 'year', 'total_output']

firm_totals['total_input'] = df.groupby(['firm_id', 'year']).apply(
    lambda x: x.loc[x['is_output'] == 0, 'v'].sum()
).values

# 3.2 外包产品的产出和投入
outsourcing_output = input_df[input_df['is_outsourcing']].groupby(['firm_id', 'year'])['v'].sum().reset_index()
outsourcing_output.columns = ['firm_id', 'year', 'outsourcing_output']

outsourcing_input = input_df[input_df['is_outsourcing']].groupby(['firm_id', 'year'])['v'].sum().reset_index()
outsourcing_input.columns = ['firm_id', 'year', 'outsourcing_input']

# 3.3 产品数量统计
product_counts = output_df.groupby(['firm_id', 'year'])['product_id'].nunique().reset_index()
product_counts.columns = ['firm_id', 'year', 'n_products']

outsourcing_product_counts = output_df[output_df['is_outsourcing']].groupby(['firm_id', 'year'])['product_id'].nunique().reset_index()
outsourcing_product_counts.columns = ['firm_id', 'year', 'n_outsourcing_products']

# 3.4 主要产出产品（销售额最大）
main_product = output_df.groupby(['firm_id', 'year', 'product_id'])['v'].sum().reset_index()
main_product = main_product.sort_values(['firm_id', 'year', 'v'], ascending=[True, True, False])
main_product = main_product.groupby(['firm_id', 'year']).first().reset_index()
main_product = main_product[['firm_id', 'year', 'product_id', 'v']]
main_product.columns = ['firm_id', 'year', 'main_product', 'main_product_sales']

# 3.5 主要外包产品（外包销售额最大）
main_outsourcing = input_df[input_df['is_outsourcing']].groupby(['firm_id', 'year', 'product_id'])['v'].sum().reset_index()
main_outsourcing = main_outsourcing.sort_values(['firm_id', 'year', 'v'], ascending=[True, True, False])
main_outsourcing = main_outsourcing.groupby(['firm_id', 'year']).first().reset_index()
main_outsourcing = main_outsourcing[['firm_id', 'year', 'product_id', 'v']]
main_outsourcing.columns = ['firm_id', 'year', 'main_outsourcing_product', 'main_outsourcing_sales']

# 合并所有信息
firm_agg = firm_totals.copy()
firm_agg = firm_agg.merge(outsourcing_output, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(outsourcing_input, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(product_counts, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(outsourcing_product_counts, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(main_product, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(main_outsourcing, on=['firm_id', 'year'], how='left')

# 填充缺失值
firm_agg['outsourcing_output'] = firm_agg['outsourcing_output'].fillna(0)
firm_agg['outsourcing_input'] = firm_agg['outsourcing_input'].fillna(0)
firm_agg['n_outsourcing_products'] = firm_agg['n_outsourcing_products'].fillna(0)

# 计算外包强度
firm_agg['outsourcing_intensity'] = firm_agg['outsourcing_output'] / firm_agg['total_output']
firm_agg['outsourcing_intensity'] = firm_agg['outsourcing_intensity'].fillna(0)

# 3.6 识别中间商（外包产品占产出>90%）
firm_agg['is_intermediary'] = (firm_agg['outsourcing_intensity'] > 0.90).astype(int)

print(f"\n企业-年份观测数: {len(firm_agg):,}")
print(f"有外包行为的企业-年份观测数: {(firm_agg['outsourcing_intensity'] > 0).sum():,}")
print(f"中间商企业-年份观测数: {firm_agg['is_intermediary'].sum():,}")
print(f"中间商占比: {firm_agg['is_intermediary'].mean()*100:.2f}%")

# 保存汇总表
firm_agg.to_stata('firm_aggregate_table.dta', write_index=False)
print("\n企业汇总表已保存至: firm_aggregate_table.dta")

# 显示样本
print("\n企业汇总表样本:")
print(firm_agg.head(10).to_string())

print("\n" + "=" * 80)
print("Step 3 完成！")
print("=" * 80)


In [ ]:
# [replicate glue] descriptive_analysis / sample 读取的 summary.dta 即上面的 firm_aggregate_table
firm_agg.to_stata('summary.dta', write_index=False)
print('saved summary.dta (= firm_aggregate_table)')

## §4　descriptive_analysis：外包普遍率 / 强度 / 相关性

代码来自你原 `descriptive_analysis.ipynb`（未改）。读 `summary.dta`，产出外包普遍率（分年）、强度分布、相关性、汇总统计等 .dta/.csv/.png。

In [ ]:
"""
企业外包行为描述性分析 - Part 2: 描述性统计
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')
os.chdir(DATA)

# 读取企业汇总表
firm_agg = pd.read_stata('summary.dta')
print("读取企业汇总表完成")
print(f"数据规模: {len(firm_agg):,} 行")



# ============================================================================
# Descriptive Fact #2: 外包的普遍性和强度
# ============================================================================
print("\n" + "=" * 80)
print("Descriptive Fact #2: 外包的普遍性和强度")
print("=" * 80)

# 2.1 按年份统计有外包行为的企业占比
firm_agg['has_outsourcing'] = (firm_agg['outsourcing_intensity'] > 0).astype(int)

outsourcing_prevalence = firm_agg.groupby('year').agg({
    'firm_id': 'nunique',
    'has_outsourcing': 'sum'
}).reset_index()
outsourcing_prevalence.columns = ['year', 'total_firms', 'firms_with_outsourcing']
outsourcing_prevalence['share_with_outsourcing'] = \
    outsourcing_prevalence['firms_with_outsourcing'] / outsourcing_prevalence['total_firms']

print("\n外包企业占比（按年份）:")
print(outsourcing_prevalence.to_string(index=False))

# 保存表格
outsourcing_prevalence.to_stata('outsourcing_prevalence.dta', write_index=False)

# 2.2 外包收入份额的分布（剔除中间商）
firm_agg_no_intermediary = firm_agg[firm_agg['is_intermediary'] == 0].copy()

# 只看有外包的企业
outsourcing_firms = firm_agg_no_intermediary[firm_agg_no_intermediary['has_outsourcing'] == 1].copy()

print(f"\n外包企业数量（剔除中间商后）: {len(outsourcing_firms):,}")
print(f"外包强度统计:")
print(outsourcing_firms['outsourcing_intensity'].describe())

# 绘制外包强度的密度分布图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：外包收入份额分布
axes[0].hist(outsourcing_firms['outsourcing_intensity'], bins=50, 
             density=True, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Outsourcing Revenue Share', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Distribution of Outsourcing Revenue Share\n(Excluding Intermediaries)', fontsize=13)
axes[0].grid(True, alpha=0.3)

# 添加均值线
mean_val = outsourcing_firms['outsourcing_intensity'].mean()
axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')
axes[0].legend()

# 右图：外包投入份额分布
outsourcing_firms['outsourcing_input_share'] = \
    outsourcing_firms['outsourcing_input'] / outsourcing_firms['total_input']
outsourcing_firms['outsourcing_input_share'] = \
    outsourcing_firms['outsourcing_input_share'].fillna(0)

axes[1].hist(outsourcing_firms['outsourcing_input_share'], bins=50, 
             density=True, alpha=0.7, edgecolor='black', color='orange')
axes[1].set_xlabel('Outsourcing Input Share', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Distribution of Outsourcing Input Share\n(Excluding Intermediaries)', fontsize=13)
axes[1].grid(True, alpha=0.3)

# 添加均值线
mean_val2 = outsourcing_firms['outsourcing_input_share'].mean()
axes[1].axvline(mean_val2, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val2:.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('outsourcing_intensity_distribution.png', dpi=300, bbox_inches='tight')
print("\n外包强度分布图已保存")

# 2.3 按百分位数统计
percentiles = [10, 25, 50, 75, 90, 95, 99]
outsourcing_percentiles = outsourcing_firms['outsourcing_intensity'].quantile(
    [p/100 for p in percentiles]
).reset_index()
outsourcing_percentiles.columns = ['percentile', 'value']
outsourcing_percentiles['percentile'] = [f'P{p}' for p in percentiles]

print("\n外包强度分位数:")
print(outsourcing_percentiles.to_string(index=False))
outsourcing_percentiles.to_csv('outsourcing_intensity_percentiles.csv', index=False)

# ============================================================================
# Descriptive Fact #1: 分销不匹配（剔除中间商）
# ============================================================================
print("\n" + "=" * 80)
print("Descriptive Fact #1: 真实生产 vs 销售的分布差异")
print("=" * 80)

# 读取原始交易数据（这里需要重新加载）
# 由于原始数据太大，这部分需要在有原始数据时运行
print("\n注意：此部分需要原始交易数据，需要单独运行")

# 伪代码逻辑：
"""
# 1. 识别非中间商企业
non_intermediary_firms = firm_agg[firm_agg['is_intermediary'] == 0]['firm_id'].unique()

# 2. 筛选这些企业的交易数据
df_non_intermediary = df[df['firm_id'].isin(non_intermediary_firms)]

# 3. 区分"真实生产"和"销售"
# 真实生产 = 只卖出、不买入的产品
df_output = df_non_intermediary[df_non_intermediary['is_output'] == 1]
df_production = df_output[~df_output['is_outsourcing']]  # 非外包产品
df_sales = df_output  # 所有销售

# 4. 计算产品层面的分布
production_by_product = df_production.groupby('product_id')['v'].sum()
sales_by_product = df_sales.groupby('product_id')['v'].sum()

# 5. 计算比率
ratio_df = pd.DataFrame({
    'production': production_by_product,
    'sales': sales_by_product
}).fillna(0)
ratio_df['ratio'] = ratio_df['sales'] / (ratio_df['production'] + 1e-10)

# 6. 绘制分布图
"""

# ============================================================================
# Descriptive Fact #3: 绩效相关性分析（部分）
# ============================================================================
print("\n" + "=" * 80)
print("Descriptive Fact #3: 绩效相关性分析")
print("=" * 80)

# 3b. 外包品销售额与主营产品销售额的相关性
correlation_data = firm_agg_no_intermediary[
    (firm_agg_no_intermediary['has_outsourcing'] == 1) &
    (firm_agg_no_intermediary['main_product_sales'].notna())
].copy()

if len(correlation_data) > 0:
    corr_3b = correlation_data[['outsourcing_output', 'main_product_sales']].corr()
    print("\n3(b) 外包品销售额 vs 主营产品销售额:")
    print(corr_3b)
    
    # 绘制散点图
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(correlation_data['main_product_sales'], 
               correlation_data['outsourcing_output'], 
               alpha=0.3, s=20)
    ax.set_xlabel('Main Product Sales (log scale)', fontsize=12)
    ax.set_ylabel('Outsourcing Output (log scale)', fontsize=12)
    ax.set_title('Correlation: Outsourcing vs Main Product Sales', fontsize=13)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # 添加拟合线
    from scipy.stats import linregress
    log_x = np.log(correlation_data['main_product_sales'] + 1)
    log_y = np.log(correlation_data['outsourcing_output'] + 1)
    slope, intercept, r_value, p_value, std_err = linregress(log_x, log_y)
    
    x_line = np.linspace(log_x.min(), log_x.max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(np.exp(x_line), np.exp(y_line), 'r-', linewidth=2, 
            label=f'R² = {r_value**2:.3f}')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('correlation_outsourcing_main_product.png', 
                dpi=300, bbox_inches='tight')
    print("相关性图已保存")

# 3d. 投入品外包份额与产出品外包份额的相关性
correlation_data_3d = firm_agg_no_intermediary[
    (firm_agg_no_intermediary['has_outsourcing'] == 1) &
    (firm_agg_no_intermediary['total_input'] > 0)
].copy()

if len(correlation_data_3d) > 0:
    corr_3d = correlation_data_3d[['outsourcing_intensity', 'outsourcing_input_share']].corr()
    print("\n3(d) 产出品外包份额 vs 投入品外包份额:")
    print(corr_3d)
    
    # 绘制散点图
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(correlation_data_3d['outsourcing_input_share'], 
               correlation_data_3d['outsourcing_intensity'], 
               alpha=0.3, s=20)
    ax.set_xlabel('Input Outsourcing Share', fontsize=12)
    ax.set_ylabel('Output Outsourcing Share', fontsize=12)
    ax.set_title('Correlation: Input vs Output Outsourcing Share', fontsize=13)
    ax.grid(True, alpha=0.3)
    
    # 添加45度线
    max_val = max(correlation_data_3d['outsourcing_input_share'].max(),
                  correlation_data_3d['outsourcing_intensity'].max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, alpha=0.5, label='45° line')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('correlation_input_output_outsourcing.png', 
                dpi=300, bbox_inches='tight')
    print("相关性图已保存")

# ============================================================================
# 汇总统计表
# ============================================================================
print("\n" + "=" * 80)
print("生成汇总统计表")
print("=" * 80)

summary_stats = pd.DataFrame({
    'Variable': [
        'Total Output',
        'Total Input', 
        'Outsourcing Output',
        'Outsourcing Input',
        'Outsourcing Intensity',
        'Number of Products',
        'Number of Outsourcing Products'
    ],
    'Mean': [
        firm_agg_no_intermediary['total_output'].mean(),
        firm_agg_no_intermediary['total_input'].mean(),
        firm_agg_no_intermediary['outsourcing_output'].mean(),
        firm_agg_no_intermediary['outsourcing_input'].mean(),
        firm_agg_no_intermediary['outsourcing_intensity'].mean(),
        firm_agg_no_intermediary['n_products'].mean(),
        firm_agg_no_intermediary['n_outsourcing_products'].mean()
    ],
    'Std': [
        firm_agg_no_intermediary['total_output'].std(),
        firm_agg_no_intermediary['total_input'].std(),
        firm_agg_no_intermediary['outsourcing_output'].std(),
        firm_agg_no_intermediary['outsourcing_input'].std(),
        firm_agg_no_intermediary['outsourcing_intensity'].std(),
        firm_agg_no_intermediary['n_products'].std(),
        firm_agg_no_intermediary['n_outsourcing_products'].std()
    ],
    'Min': [
        firm_agg_no_intermediary['total_output'].min(),
        firm_agg_no_intermediary['total_input'].min(),
        firm_agg_no_intermediary['outsourcing_output'].min(),
        firm_agg_no_intermediary['outsourcing_input'].min(),
        firm_agg_no_intermediary['outsourcing_intensity'].min(),
        firm_agg_no_intermediary['n_products'].min(),
        firm_agg_no_intermediary['n_outsourcing_products'].min()
    ],
    'Max': [
        firm_agg_no_intermediary['total_output'].max(),
        firm_agg_no_intermediary['total_input'].max(),
        firm_agg_no_intermediary['outsourcing_output'].max(),
        firm_agg_no_intermediary['outsourcing_input'].max(),
        firm_agg_no_intermediary['outsourcing_intensity'].max(),
        firm_agg_no_intermediary['n_products'].max(),
        firm_agg_no_intermediary['n_outsourcing_products'].max()
    ]
})

print("\n描述性统计（剔除中间商）:")
print(summary_stats.to_string(index=False))

summary_stats.to_stata('summary_statistics.dta', write_index=False)

print("\n" + "=" * 80)
print("所有分析完成！")
print("=" * 80)


## §5　coverage：选择集覆盖 + 相似度分布

代码来自你原 `coverage.ipynb`（未改）。读 `full_data` + similarity；**主产品这里重算为 production_value 最大**；算 top30/50/100 的产品数/销售额覆盖率 → `coverage_summary.csv`，并画相似度分布图。

In [ ]:
"""
Coverage Diagnostics & Similarity Distribution Analysis
========================================================
Tasks:
  1. Coverage of top30/top50/top100 choice sets (product count & sales share)
  2. Similarity distribution of secondary products (count & share-weighted)

Input files (in working directory):
  - full_data.dta
  - full_product_similarity.dta

Output: figures + summary tables in diagnostics/ folder
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
WORK_DIR = str(DATA)
OUT_DIR = os.path.join(WORK_DIR, "diagnostics")
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(WORK_DIR)

TOP_N_LIST = [30, 50, 100]
NBINS = 50  # number of bins for histograms

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})


# ============================================================
# PART 0: Load data
# ============================================================
print("=" * 60)
print("Loading data...")
print("=" * 60)

# --- Load full_data ---
print("  Reading full_data.dta ...")
df = pd.read_stata("full_data.dta",
                    columns=['firm_id', 'year', 'product_id',
                             'is_intermediary', 'production_value',
                             'total_output', 'outsourcing_percen'])
print(f"  full_data: {len(df):,} rows")

# Drop intermediaries
df = df[df['is_intermediary'] != 1].copy()
df.drop(columns=['is_intermediary'], inplace=True)
print(f"  After dropping intermediaries: {len(df):,} rows")

# --- Define main product = max production_value per firm×year ---
df['prod_rank'] = df.groupby(['firm_id', 'year'])['production_value'] \
                    .rank(method='first', ascending=False)
df['is_main'] = (df['prod_rank'] == 1).astype(int)

# --- Separate main and secondary ---
main_df = df[df['is_main'] == 1][['firm_id', 'year', 'product_id']].copy()
main_df.rename(columns={'product_id': 'main_pid'}, inplace=True)

sec_df = df[df['is_main'] == 0].copy()
sec_df = sec_df.merge(main_df, on=['firm_id', 'year'], how='left')
# total_output here is the secondary product's sales
sec_df.rename(columns={'total_output': 'sec_sales'}, inplace=True)

# Compute total secondary sales per firm×year
sec_df['total_sec_sales'] = sec_df.groupby(['firm_id', 'year'])['sec_sales'] \
                                  .transform('sum')
sec_df['sec_sales_share'] = sec_df['sec_sales'] / sec_df['total_sec_sales']
sec_df['sec_sales_share'] = sec_df['sec_sales_share'].fillna(0)

print(f"  Main products: {len(main_df):,}")
print(f"  Secondary products: {len(sec_df):,}")

del df  # free memory

# --- Load similarity matrix ---
print("  Reading full_product_similarity.dta ...")
sim_df = pd.read_stata(str(SRC/'full_product_similarity.dta'))
print(f"  Similarity pairs (unidirectional): {len(sim_df):,}")

# Make bidirectional
sim_rev = sim_df.rename(columns={'product_1': 'product_2', 'product_2': 'product_1'})
sim_bi = pd.concat([sim_df, sim_rev], ignore_index=True)
sim_bi.drop_duplicates(subset=['product_1', 'product_2'], inplace=True)
sim_bi.rename(columns={'product_1': 'main_pid', 'product_2': 'product_id'},
              inplace=True)
print(f"  Bidirectional pairs: {len(sim_bi):,}")

del sim_df, sim_rev


# ============================================================
# PART 1: Merge similarity to secondary products
# ============================================================
print("\n" + "=" * 60)
print("Merging similarity to secondary products...")
print("=" * 60)

sec_df['main_pid'] = sec_df['main_pid'].astype(str).str.strip()
sec_df['product_id'] = sec_df['product_id'].astype(str).str.strip()
sim_bi['main_pid'] = sim_bi['main_pid'].astype(str).str.strip()
sim_bi['product_id'] = sim_bi['product_id'].astype(str).str.strip()

sec_with_sim = sec_df.merge(
    sim_bi[['main_pid', 'product_id', 'input_similarity', 'output_similarity']],
    on=['main_pid', 'product_id'],
    how='left'
)

n_total = len(sec_with_sim)
n_matched = sec_with_sim['input_similarity'].notna().sum()
print(f"  Total secondary products: {n_total:,}")
print(f"  Matched with similarity: {n_matched:,} ({n_matched/n_total*100:.1f}%)")
print(f"  Unmatched (same as main or missing): {n_total - n_matched:,}")

# Drop unmatched (product_id == main_pid or not in similarity matrix)
sec_with_sim = sec_with_sim.dropna(subset=['input_similarity'])
print(f"  Working sample: {len(sec_with_sim):,}")


# ============================================================
# PART 2: Build choice sets for top30/50/100 and compute coverage
# ============================================================
print("\n" + "=" * 60)
print("Building choice sets and computing coverage...")
print("=" * 60)

# Rank products by input_sim and output_sim for each main product
print("  Ranking products per main product...")
ranks = sim_bi.copy()
ranks['rank_input'] = ranks.groupby('main_pid')['input_similarity'] \
                           .rank(method='first', ascending=False)
ranks['rank_output'] = ranks.groupby('main_pid')['output_similarity'] \
                            .rank(method='first', ascending=False)

coverage_results = []

for top_n in TOP_N_LIST:
    print(f"\n  --- Top {top_n} ---")

    # Choice set = input top N ∪ output top N
    choice = ranks[(ranks['rank_input'] <= top_n) |
                   (ranks['rank_output'] <= top_n)].copy()
    choice = choice[['main_pid', 'product_id']].drop_duplicates()

    n_pairs = len(choice)
    avg_candidates = choice.groupby('main_pid').size().mean()
    print(f"    Total pairs: {n_pairs:,}, avg candidates/main: {avg_candidates:.1f}")

    # Mark which secondary products are in the choice set
    choice['in_topN'] = 1
    merged = sec_with_sim.merge(
        choice, on=['main_pid', 'product_id'], how='left'
    )
    merged['in_topN'] = merged['in_topN'].fillna(0).astype(int)

    # --- Coverage by count ---
    n_covered = merged['in_topN'].sum()
    n_all = len(merged)
    count_coverage = n_covered / n_all

    # --- Coverage by secondary sales share ---
    # For each firm×year, what share of secondary sales is covered?
    covered_sales = merged.loc[merged['in_topN'] == 1, 'sec_sales'].sum()
    total_sales = merged['sec_sales'].sum()
    sales_coverage = covered_sales / total_sales

    print(f"    Product count coverage: {count_coverage*100:.1f}%")
    print(f"    Sales share coverage:   {sales_coverage*100:.1f}%")

    coverage_results.append({
        'top_N': top_n,
        'n_pairs': n_pairs,
        'avg_candidates': avg_candidates,
        'count_coverage': count_coverage,
        'sales_coverage': sales_coverage,
    })

    del choice, merged

# Save coverage summary
cov_df = pd.DataFrame(coverage_results)
cov_df.to_csv(os.path.join(OUT_DIR, "coverage_summary.csv"), index=False)
print(f"\n  Coverage summary saved to diagnostics/coverage_summary.csv")
print(cov_df.to_string(index=False))

del ranks


# ============================================================
# PART 3: Similarity distributions of secondary products
# ============================================================
print("\n" + "=" * 60)
print("Plotting similarity distributions...")
print("=" * 60)

for sim_var, sim_label in [('input_similarity', 'Input Similarity'),
                           ('output_similarity', 'Output Similarity')]:

    vals = sec_with_sim[sim_var].values
    weights = sec_with_sim['sec_sales_share'].values
    sales = sec_with_sim['sec_sales'].values

    # Normalize sales weights to sum to 1
    sales_total = sales.sum()
    sales_weights = sales / sales_total if sales_total > 0 else sales

    # ----------------------------------------------------------
    # Figure A: Count (unweighted) histogram
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(vals, bins=NBINS, color='steelblue', edgecolor='white',
            alpha=0.85)
    ax.set_xlabel(sim_label)
    ax.set_ylabel('Number of Secondary Products')
    ax.set_title(f'Distribution of {sim_label} — Unweighted (Count)')
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0, 0))
    fig.tight_layout()
    fname = f"dist_{sim_var}_count.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # ----------------------------------------------------------
    # Figure B: Weighted by within-firm secondary sales share
    #   (each product's share of its firm×year's total secondary sales)
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(vals, bins=NBINS, weights=weights,
            color='darkorange', edgecolor='white', alpha=0.85)
    ax.set_xlabel(sim_label)
    ax.set_ylabel('Cumulative Within-Firm Secondary Sales Share')
    ax.set_title(f'Distribution of {sim_label} — Weighted by Secondary Sales Share')
    fig.tight_layout()
    fname = f"dist_{sim_var}_share_within.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # ----------------------------------------------------------
    # Figure C: Weighted by global secondary sales
    #   (each product's sales / total secondary sales across all firms)
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(vals, bins=NBINS, weights=sales_weights,
            color='forestgreen', edgecolor='white', alpha=0.85)
    ax.set_xlabel(sim_label)
    ax.set_ylabel('Share of Total Secondary Sales')
    ax.set_title(f'Distribution of {sim_label} — Weighted by Total Secondary Sales')
    fig.tight_layout()
    fname = f"dist_{sim_var}_share_global.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # ----------------------------------------------------------
    # Figure D: Overlay — count vs share-weighted (normalized to density)
    # ----------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Count density (left axis)
    ax1.hist(vals, bins=NBINS, density=True,
             color='steelblue', edgecolor='white', alpha=0.5,
             label='Count (density)')
    ax1.set_xlabel(sim_label)
    ax1.set_ylabel('Density (Count)', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')

    # Share-weighted density (right axis)
    ax2 = ax1.twinx()
    ax2.hist(vals, bins=NBINS, weights=weights, density=True,
             color='darkorange', edgecolor='white', alpha=0.5,
             label='Sales-Share Weighted (density)')
    ax2.set_ylabel('Density (Sales-Share Weighted)', color='darkorange')
    ax2.tick_params(axis='y', labelcolor='darkorange')

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    ax1.set_title(f'{sim_label}: Count vs Sales-Share Weighted Density')
    fig.tight_layout()
    fname = f"dist_{sim_var}_overlay.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")


# ============================================================
# PART 4: Coverage by similarity bins (for visual diagnostics)
# ============================================================
print("\n" + "=" * 60)
print("Coverage by similarity bins...")
print("=" * 60)

for sim_var, sim_label in [('input_similarity', 'Input Similarity'),
                           ('output_similarity', 'Output Similarity')]:

    # Re-merge with top N flags
    for top_n in TOP_N_LIST:
        choice = sim_bi.copy()
        choice['rank_input'] = choice.groupby('main_pid')['input_similarity'] \
                                     .rank(method='first', ascending=False)
        choice['rank_output'] = choice.groupby('main_pid')['output_similarity'] \
                                      .rank(method='first', ascending=False)
        choice = choice[(choice['rank_input'] <= top_n) |
                        (choice['rank_output'] <= top_n)]
        choice = choice[['main_pid', 'product_id']].drop_duplicates()
        choice[f'in_top{top_n}'] = 1
        sec_with_sim = sec_with_sim.merge(
            choice, on=['main_pid', 'product_id'], how='left'
        )
        sec_with_sim[f'in_top{top_n}'] = sec_with_sim[f'in_top{top_n}'].fillna(0)
        del choice

    # Bin by similarity
    sec_with_sim[f'{sim_var}_bin'] = pd.cut(
        sec_with_sim[sim_var], bins=20, include_lowest=True
    )

    bin_stats = []
    for bn, grp in sec_with_sim.groupby(f'{sim_var}_bin', observed=True):
        row = {
            'bin': str(bn),
            'bin_mid': bn.mid,
            'count': len(grp),
            'total_sales': grp['sec_sales'].sum(),
        }
        for top_n in TOP_N_LIST:
            covered = grp[f'in_top{top_n}']
            row[f'count_cov_top{top_n}'] = covered.sum() / len(grp) if len(grp) > 0 else 0
            covered_sales = grp.loc[covered == 1, 'sec_sales'].sum()
            row[f'sales_cov_top{top_n}'] = covered_sales / grp['sec_sales'].sum() \
                if grp['sec_sales'].sum() > 0 else 0
        bin_stats.append(row)

    bin_df = pd.DataFrame(bin_stats)

    # Plot coverage rate by similarity bin
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    for top_n in TOP_N_LIST:
        ax1.plot(bin_df['bin_mid'], bin_df[f'count_cov_top{top_n}'],
                 marker='o', markersize=4, label=f'Top {top_n}')
        ax2.plot(bin_df['bin_mid'], bin_df[f'sales_cov_top{top_n}'],
                 marker='o', markersize=4, label=f'Top {top_n}')

    ax1.set_xlabel(sim_label)
    ax1.set_ylabel('Product Count Coverage Rate')
    ax1.set_title(f'Count Coverage by {sim_label} Bin')
    ax1.legend()
    ax1.set_ylim(-0.05, 1.05)

    ax2.set_xlabel(sim_label)
    ax2.set_ylabel('Sales Share Coverage Rate')
    ax2.set_title(f'Sales Coverage by {sim_label} Bin')
    ax2.legend()
    ax2.set_ylim(-0.05, 1.05)

    fig.tight_layout()
    fname = f"coverage_by_{sim_var}_bin.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # Drop temp columns
    for top_n in TOP_N_LIST:
        sec_with_sim.drop(columns=[f'in_top{top_n}'], inplace=True, errors='ignore')
    sec_with_sim.drop(columns=[f'{sim_var}_bin'], inplace=True, errors='ignore')


# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 60)
print("All done! Output in diagnostics/ folder:")
print("=" * 60)
print("  coverage_summary.csv")
print("  dist_input_similarity_count.png")
print("  dist_input_similarity_share_within.png")
print("  dist_input_similarity_share_global.png")
print("  dist_input_similarity_overlay.png")
print("  dist_output_similarity_count.png")
print("  dist_output_similarity_share_within.png")
print("  dist_output_similarity_share_global.png")
print("  dist_output_similarity_overlay.png")
print("  coverage_by_input_similarity_bin.png")
print("  coverage_by_output_similarity_bin.png")

## §6　sample：企业级余弦相似度 → similarity.dta

代码来自你原 `sample.ipynb`（未改，5 格）。从 `summary.dta` 抽 5 万非中介外包企业（2018），用 `io_table_lite` 把外包品/自产品分别加权成投入向量，算两者余弦相似度 → `similarity.dta`。

In [ ]:
%reset -f
# [replicate paths] 代码在 Empirical1(git 共享)；生成数据全部落到 DATA
from pathlib import Path
import os
DATA = Path(r'G:\Kuangyu_Temp\Outsource\replicate')   # 所有生成数据（只在 VM）
SRC  = Path(r'G:\Kuangyu_Temp\Outsource')              # 已有输入数据（similarity / io_table）
DATA.mkdir(exist_ok=True)
os.chdir(DATA)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
from scipy.spatial.distance import cosine
warnings.filterwarnings('ignore')
os.chdir(DATA)
# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置绘图风格
sns.set_style("whitegrid")
sns.set_palette("husl")

# 读取数据
firm_agg = pd.read_stata('summary.dta')
firm_agg = firm_agg[firm_agg['year'] == 2018]
df1 = pd.read_stata(str(DATA/'lenth9_18.dta'))

In [ ]:
np.random.seed(42)


selected = firm_agg[(firm_agg['is_intermediary'] == 0)&(firm_agg['outsourcing_intensity']>0)]
sampled_firms = selected.sample(n = min(50000, len(selected)), random_state=42)['firm_id'].values
sampled_firms = [str(x) for x in sampled_firms]

In [ ]:
df = df1[df1['firm_id'].isin(sampled_firms)]

buy = df[df['is_output'] == 0].groupby(['firm_id', 'product_id'])['v'].sum().reset_index()
buy.columns = ['firm_id', 'product_id', 'buy']

sell = df[df['is_output'] == 1].groupby(['firm_id', 'product_id'])['v'].sum().reset_index()
sell.columns = ['firm_id', 'product_id', 'sell']

prod = sell.merge(buy, on = ['firm_id', 'product_id'], how = 'left').fillna(0)
prod['outsource'] = prod[['buy', 'sell']].min(axis = 1)
prod['own'] = prod['sell'] - prod['outsource']

In [ ]:
io = pd.read_stata(str(SRC/'io_table_lite.dta'))

results = []

for firm in sampled_firms:
    fp = prod[prod['firm_id'] == firm]
    
    outsource_prod = fp[fp['outsource']>0][['product_id', 'outsource']]
    own_prod = fp[fp['own']>0][['product_id', 'own']]
    
    if len(outsource_prod) == 0 or len(own_prod) == 0:
        continue
    
    io_out = io[io['product_id'].isin(outsource_prod['product_id'])].copy()
    io_out = io_out.merge(outsource_prod, on = 'product_id')
    vec_out = io_out.groupby('product_id_input').apply(
        lambda x: (x['coefficient']*x['outsource']).sum()
    )/outsource_prod['outsource'].sum()
    
    io_own = io[io['product_id'].isin(own_prod['product_id'])].copy()
    io_own = io_own.merge(own_prod, on = 'product_id')
    vec_own = io_own.groupby('product_id_input').apply(
        lambda x: (x['coefficient']*x['own']).sum()
    )/own_prod['own'].sum()
    
    #统一维度
    all_inputs = sorted(set(vec_out.index) | set(vec_own.index))
    v1 = [vec_out.get(i, 0) for i in all_inputs]
    v2 = [vec_own.get(i, 0) for i in all_inputs]
    
    if sum(v1) >0 and sum(v2)>0:
        sim = 1-cosine(v1, v2)
        results.append({'firm_id': firm, 'cosine_similarity':sim})

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_stata('similarity.dta', write_index = False)